In [32]:
import re
import os
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook
from collections import defaultdict
from datetime import datetime

# ============================================================
# Input files
# ============================================================

summary_csv = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/Clinical_data/"
    "Summary_TreatmentLine_TPNumber_FINAL_filled_gap_assigned.csv"
)

mastersheet_xlsx = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/NST/annotate_apollo_nst_treatment_line/"
    "APOLLO_NST_mastersheet.xlsx"
)

sheet_name = "Seq_Stat"

# ============================================================
# Output files 
# ============================================================

out_xlsx = mastersheet_xlsx.with_name(
    "APOLLO_NST_mastersheet_Seq_Stat_with_TreatmentLine.xlsx"
)

audit_csv = mastersheet_xlsx.with_name(
    "APOLLO_NST_Seq_Stat_TreatmentLine_mapping_audit.csv"
)

unmatched_csv = mastersheet_xlsx.with_name(
    "APOLLO_NST_Seq_Stat_TreatmentLine_unmatched.csv"
)

conflict_csv = mastersheet_xlsx.with_name(
    "APOLLO_NST_Seq_Stat_TreatmentLine_conflicts.csv"
)

# ============================================================
# Check output directory
# ============================================================

out_dir = mastersheet_xlsx.parent

print("Output directory:", out_dir)
print("Directory exists:", out_dir.exists())
print("Directory writable:", os.access(out_dir, os.W_OK))

if not out_dir.exists():
    raise FileNotFoundError(f"Output directory does not exist: {out_dir}")

if not os.access(out_dir, os.W_OK):
    raise PermissionError(f"No write permission for output directory: {out_dir}")

# Test write permission
test_file = out_dir / "__write_test__.txt"
test_file.write_text("write test OK\n")
test_file.unlink()

print("Write test passed")

# ============================================================
# Helper functions
# ============================================================

def normalize_header(x):
    return str(x).strip().lower()


def ordinal_line(n):
    if 10 <= n % 100 <= 20:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix} line"


def extract_tp_entries(value):
    """
    Extract TP entries:
    TP21-M146 (6) -> base: TP21-M146, full: TP21-M146 (6)
    TP19-M314    (5) -> base: TP19-M314, full: TP19-M314 (5)
    TP20-M220(5) -> base: TP20-M220, full: TP20-M220 (5)
    """
    if pd.isna(value):
        return []

    text = str(value).strip()
    text = text.replace("\u00A0", " ")
    text = text.replace("–", "-").replace("—", "-").replace("−", "-")

    pattern = re.compile(
        r"(TP\s*\d{2,4}\s*-\s*M\s*\d+)\s*(?:\(\s*([^)]+?)\s*\))?",
        flags=re.IGNORECASE
    )

    entries = []

    for m in pattern.finditer(text):
        raw_base = m.group(1)
        raw_paren = m.group(2)

        base = re.sub(r"\s+", "", raw_base.upper())

        if raw_paren is not None:
            paren = re.sub(r"\s+", "", str(raw_paren))
            full = f"{base} ({paren})"
        else:
            full = base

        entries.append({
            "base": base,
            "full": full
        })

    # Remove duplicates
    seen = set()
    unique_entries = []

    for e in entries:
        key = (e["base"], e["full"])
        if key not in seen:
            unique_entries.append(e)
            seen.add(key)

    return unique_entries

# ============================================================
# Step 1: Read treatment-line summary and build maps
# ============================================================

summary_df = pd.read_csv(summary_csv, dtype=str)

full_tp_to_lines = defaultdict(set)
base_tp_to_lines = defaultdict(set)

for line_num in range(1, 18):
    line_label = ordinal_line(line_num)
    col = f"TP Number {line_label}"

    if col not in summary_df.columns:
        print(f"WARNING: Column not found: {col}")
        continue

    for idx, value in summary_df[col].items():
        entries = extract_tp_entries(value)

        for e in entries:
            full_tp_to_lines[e["full"]].add(line_label)
            base_tp_to_lines[e["base"]].add(line_label)

print("\nTreatment-line maps created")
print(f"Unique full TP keys mapped: {len(full_tp_to_lines)}")
print(f"Unique base TP keys mapped: {len(base_tp_to_lines)}")

# ============================================================
# Step 2: Open workbook and Seq_Stat sheet
# ============================================================

wb = load_workbook(mastersheet_xlsx)

if sheet_name not in wb.sheetnames:
    raise ValueError(f"Sheet not found: {sheet_name}. Available sheets: {wb.sheetnames}")

ws = wb[sheet_name]

header_row = 1
headers = {}

for cell in ws[header_row]:
    if cell.value is not None:
        headers[normalize_header(cell.value)] = cell.column

sample_col_name = normalize_header("Sample ID (TP number)")
treatment_col_name = normalize_header("Treatment Line")

if sample_col_name not in headers:
    raise ValueError(
        f"Could not find column: Sample ID (TP number). "
        f"Available columns: {[cell.value for cell in ws[header_row] if cell.value is not None]}"
    )

sample_col = headers[sample_col_name]

if treatment_col_name in headers:
    treatment_col = headers[treatment_col_name]
else:
    treatment_col = ws.max_column + 1
    ws.cell(row=header_row, column=treatment_col).value = "Treatment Line"

print("\nWorkbook loaded")
print(f"Using sheet: {sheet_name}")
print(f"Sample ID column number: {sample_col}")
print(f"Treatment Line column number: {treatment_col}")

# ============================================================
# Step 3: Fill Treatment Line
# ============================================================

OVERWRITE_EXISTING = True

audit_rows = []
unmatched_rows = []
conflict_rows = []

matched_count = 0
unmatched_count = 0
skipped_existing_count = 0

for row in range(header_row + 1, ws.max_row + 1):
    sample_value = ws.cell(row=row, column=sample_col).value
    existing_treatment = ws.cell(row=row, column=treatment_col).value

    entries = extract_tp_entries(sample_value)

    matched_lines = []
    matched_tps = []
    match_types = []

    for e in entries:
        full_key = e["full"]
        base_key = e["base"]

        # Exact full match first, e.g. TP21-M146 (6)
        if full_key in full_tp_to_lines:
            lines = sorted(full_tp_to_lines[full_key])
            matched_lines.extend(lines)
            matched_tps.append(full_key)
            match_types.append("Full TP + parenthesis match")

        # Fallback to base TP, e.g. TP21-M146
        elif base_key in base_tp_to_lines:
            lines = sorted(base_tp_to_lines[base_key])
            matched_lines.extend(lines)
            matched_tps.append(base_key)
            match_types.append("Base TP fallback match")

    matched_lines = list(dict.fromkeys(matched_lines))
    matched_tps = list(dict.fromkeys(matched_tps))
    match_types = list(dict.fromkeys(match_types))

    if matched_lines:
        treatment_value = "; ".join(matched_lines)

        if len(matched_lines) > 1:
            conflict_rows.append({
                "Excel_Row": row,
                "Sample_ID_TP_number": sample_value,
                "Extracted_Full_TP": "; ".join([e["full"] for e in entries]),
                "Extracted_Base_TP": "; ".join([e["base"] for e in entries]),
                "Matched_TP": "; ".join(matched_tps),
                "Matched_Treatment_Lines": treatment_value,
                "Match_Type": "; ".join(match_types)
            })

        if existing_treatment not in [None, ""] and not OVERWRITE_EXISTING:
            skipped_existing_count += 1
            action = "Skipped existing value"
        else:
            ws.cell(row=row, column=treatment_col).value = treatment_value
            matched_count += 1
            action = "Filled"

        audit_rows.append({
            "Excel_Row": row,
            "Sample_ID_TP_number": sample_value,
            "Extracted_Full_TP": "; ".join([e["full"] for e in entries]),
            "Extracted_Base_TP": "; ".join([e["base"] for e in entries]),
            "Matched_TP": "; ".join(matched_tps),
            "Assigned_Treatment_Line": treatment_value,
            "Previous_Treatment_Line": existing_treatment,
            "Match_Type": "; ".join(match_types),
            "Action": action
        })

    else:
        unmatched_count += 1
        unmatched_rows.append({
            "Excel_Row": row,
            "Sample_ID_TP_number": sample_value,
            "Extracted_Full_TP": "; ".join([e["full"] for e in entries]),
            "Extracted_Base_TP": "; ".join([e["base"] for e in entries]),
            "Previous_Treatment_Line": existing_treatment
        })

print("\nTreatment-line assignment completed")
print(f"Matched / filled rows: {matched_count}")
print(f"Unmatched rows: {unmatched_count}")
print(f"Skipped existing rows: {skipped_existing_count}")
print(f"Potential multi-line conflicts: {len(conflict_rows)}")

# ============================================================
# Step 4: Save files
# ============================================================

wb.save(str(out_xlsx))

pd.DataFrame(audit_rows).to_csv(str(audit_csv), index=False)
pd.DataFrame(unmatched_rows).to_csv(str(unmatched_csv), index=False)
pd.DataFrame(conflict_rows).to_csv(str(conflict_csv), index=False)

# ============================================================
# Step 5: Verify files exist
# ============================================================

print("\nVerifying saved files")

for f in [out_xlsx, audit_csv, unmatched_csv, conflict_csv]:
    f = Path(f)

    print("=" * 80)
    print(f)

    if not f.exists():
        raise FileNotFoundError(f"File was not saved: {f}")

    print("Exists:", f.exists())
    print("Size:", f.stat().st_size, "bytes")
    print("Modified:", datetime.fromtimestamp(f.stat().st_mtime))

print("\nDONE: all output files were created successfully.")

Output directory: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/NST/annotate_apollo_nst_treatment_line
Directory exists: True
Directory writable: True
Write test passed

Treatment-line maps created
Unique full TP keys mapped: 980
Unique base TP keys mapped: 977

Workbook loaded
Using sheet: Seq_Stat
Sample ID column number: 4
Treatment Line column number: 7

Treatment-line assignment completed
Matched / filled rows: 358
Unmatched rows: 23
Skipped existing rows: 0
Potential multi-line conflicts: 2

Verifying saved files
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/NST/annotate_apollo_nst_treatment_line/APOLLO_NST_mastersheet_Seq_Stat_with_TreatmentLine.xlsx
Exists: True
Size: 83712 bytes
Modified: 2026-05-01 12:20:25.878680
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/NST/annotate_apollo_nst_treatment_line/APOLLO_NST_Seq_Stat_TreatmentLine_mapping_audit.csv
Exists: True
Size: 36536 bytes
Modified: 2026-05-01 12:20:27.621746
/

NOT FOUND ANYWHERE    21
FOUND                  5
Name: Found_Status, dtype: int64

Saved:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/NST/APOLLO_NST_Seq_Stat_unmatched_search_in_summary.csv


,Sample_ID_TP_number,Extracted_Full_TP,Extracted_Base_TP,Found_Status,Found_In_Column,Summary_Row_Index,Summary_Value
0,TP21-M146 (6),TP21-M146 (6),TP21-M146,NOT FOUND ANYWHERE,,,
1,TP21-M780 (9),TP21-M780 (9),TP21-M780,NOT FOUND ANYWHERE,,,
2,TP22-M12 (8),TP22-M12 (8),TP22-M12,FOUND,TP Number 1st line,71,TP22-M120 (3)
3,TP22-M12 (8),TP22-M12 (8),TP22-M12,FOUND,TP Number 4th line,235,TP22-M121 (5)
4,TP22-M12 (8),TP22-M12 (8),TP22-M12,FOUND,TP Number 7th line,94,TP22-M128 (8)
5,TP22-M102 (6),TP22-M102 (6),TP22-M102,NOT FOUND ANYWHERE,,,
6,TP19-M314 (5),TP19-M314 (5),TP19-M314,NOT FOUND ANYWHERE,,,
7,TP20-M531 (6),TP20-M531 (6),TP20-M531,NOT FOUND ANYWHERE,,,
8,TP20-M39 (8),TP20-M39 (8),TP20-M39,FOUND,TP Number 2nd line,72,TP20-M394 (1)
9,TP20-M39 (8),TP20-M39 (8),TP20-M39,FOUND,TP Number 3rd line,308,TP19-M948 (8);TP20-M393 (10)


In [29]:
import re
import pandas as pd
from pathlib import Path

summary_csv = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/Clinical_data/"
    "Summary_TreatmentLine_TPNumber_FINAL_filled_gap_assigned.csv"
)

unmatched_csv = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/NST/"
    "APOLLO_NST_Seq_Stat_TreatmentLine_unmatched.csv"
)

out_exact_search_csv = unmatched_csv.with_name(
    "APOLLO_NST_Seq_Stat_unmatched_exact_search_in_summary.csv"
)

summary_df = pd.read_csv(summary_csv, dtype=str)
unmatched_df = pd.read_csv(unmatched_csv, dtype=str)


def ordinal_line(n):
    if 10 <= n % 100 <= 20:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix} line"


def extract_tp_entries(value):
    """
    Exact TP extractor:
    TP22-M12 (8)  -> full: TP22-M12 (8), base: TP22-M12
    TP22-M120 (3) -> full: TP22-M120 (3), base: TP22-M120
    These remain distinct.
    """
    if pd.isna(value):
        return []

    text = str(value).strip()
    text = text.replace("\u00A0", " ")
    text = text.replace("–", "-").replace("—", "-").replace("−", "-")

    pattern = re.compile(
        r"(TP\s*\d{2,4}\s*-\s*M\s*\d+)\s*(?:\(\s*([^)]+?)\s*\))?",
        flags=re.IGNORECASE
    )

    entries = []

    for m in pattern.finditer(text):
        raw_base = m.group(1)
        raw_paren = m.group(2)

        base = re.sub(r"\s+", "", raw_base.upper())

        if raw_paren is not None:
            paren = re.sub(r"\s+", "", str(raw_paren))
            full = f"{base} ({paren})"
        else:
            full = base

        entries.append({
            "base": base,
            "full": full
        })

    # remove duplicates
    seen = set()
    unique_entries = []

    for e in entries:
        key = (e["base"], e["full"])
        if key not in seen:
            unique_entries.append(e)
            seen.add(key)

    return unique_entries


# Only search treatment-line TP columns
tp_line_cols = [
    f"TP Number {ordinal_line(i)}"
    for i in range(1, 18)
    if f"TP Number {ordinal_line(i)}" in summary_df.columns
]

exact_search_rows = []

for _, row in unmatched_df.iterrows():
    sample = row["Sample_ID_TP_number"]
    query_full = str(row["Extracted_Full_TP"]).strip().upper()
    query_base = str(row["Extracted_Base_TP"]).strip().upper()

    found = False

    for col in tp_line_cols:
        for idx, value in summary_df[col].items():
            entries = extract_tp_entries(value)

            for e in entries:
                summary_full = e["full"]
                summary_base = e["base"]

                if query_full == summary_full or query_base == summary_base:
                    found = True
                    exact_search_rows.append({
                        "Sample_ID_TP_number": sample,
                        "Query_Full_TP": query_full,
                        "Query_Base_TP": query_base,
                        "Found_Status": "EXACT FOUND",
                        "Found_In_Column": col,
                        "Summary_Row_Index": idx + 2,
                        "Summary_Value": value,
                        "Matched_Summary_Full_TP": summary_full,
                        "Matched_Summary_Base_TP": summary_base
                    })

    if not found:
        exact_search_rows.append({
            "Sample_ID_TP_number": sample,
            "Query_Full_TP": query_full,
            "Query_Base_TP": query_base,
            "Found_Status": "NOT FOUND AS EXACT TP",
            "Found_In_Column": "",
            "Summary_Row_Index": "",
            "Summary_Value": "",
            "Matched_Summary_Full_TP": "",
            "Matched_Summary_Base_TP": ""
        })

exact_search_df = pd.DataFrame(exact_search_rows)
exact_search_df.to_csv(out_exact_search_csv, index=False)

print(exact_search_df["Found_Status"].value_counts())
print("\nSaved:")
print(out_exact_search_csv)

exact_search_df

NOT FOUND AS EXACT TP    23
Name: Found_Status, dtype: int64

Saved:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/NST/APOLLO_NST_Seq_Stat_unmatched_exact_search_in_summary.csv


,Sample_ID_TP_number,Query_Full_TP,Query_Base_TP,Found_Status,Found_In_Column,Summary_Row_Index,Summary_Value,Matched_Summary_Full_TP,Matched_Summary_Base_TP
0,TP21-M146 (6),TP21-M146 (6),TP21-M146,NOT FOUND AS EXACT TP,,,,,
1,TP21-M780 (9),TP21-M780 (9),TP21-M780,NOT FOUND AS EXACT TP,,,,,
2,TP22-M12 (8),TP22-M12 (8),TP22-M12,NOT FOUND AS EXACT TP,,,,,
3,TP22-M102 (6),TP22-M102 (6),TP22-M102,NOT FOUND AS EXACT TP,,,,,
4,TP19-M314 (5),TP19-M314 (5),TP19-M314,NOT FOUND AS EXACT TP,,,,,
5,TP20-M531 (6),TP20-M531 (6),TP20-M531,NOT FOUND AS EXACT TP,,,,,
6,TP20-M39 (8),TP20-M39 (8),TP20-M39,NOT FOUND AS EXACT TP,,,,,
7,TP19-M557 (6),TP19-M557 (6),TP19-M557,NOT FOUND AS EXACT TP,,,,,
8,TP21-M279 (9),TP21-M279 (9),TP21-M279,NOT FOUND AS EXACT TP,,,,,
9,TP19-M612 (2),TP19-M612 (2),TP19-M612,NOT FOUND AS EXACT TP,,,,,
